In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression

# =============================================================================
# GENERIC UTILITY FUNCTIONS
# =============================================================================

def calculate_stats(samples, confidence=0.95):
    """
    Computes mean, variance of the estimator, and asymptotic confidence interval.
    """
    n = len(samples)
    mu = np.mean(samples)
    # Variance of the variable (unbiased)
    s2 = np.var(samples, ddof=1)
    # Variance of the estimator: Var(mean) = Var(X) / n
    var_estimator = s2 / n
    std_error = np.sqrt(var_estimator)
    
    z = stats.norm.ppf((1 + confidence) / 2)
    ci = (mu - z * std_error, mu + z * std_error)
    
    return {
        "mean": mu,
        "var_estimator": var_estimator,
        "std_error": std_error,
        "ci": ci
    }

# =============================================================================
# EXERCISE 3.1: IMPORTANCE SAMPLING FOR TAIL PROBABILITIES
# =============================================================================

def run_importance_sampling(target_pdf, proposal_dist, func, n):
    """
    Generic Importance Sampling Estimator.
    target_pdf: function(x) - PDF of the target distribution
    proposal_dist: scipy.stats distribution object (must have rvs and pdf methods)
    func: function(x) - The function we want to integrate (e.g., indicator function)
    """
    # Sample from proposal distribution q(x)
    x_samples = proposal_dist.rvs(size=n)
    
    # Calculate weights: w(x) = p(x) / q(x)
    p_vals = target_pdf(x_samples)
    q_vals = proposal_dist.pdf(x_samples)
    
    weights = p_vals / q_vals
    
    # Target value: f(x) * w(x)
    results = func(x_samples) * weights
    return calculate_stats(results)

def solve_3_1_tail_probability():
    """
    Calculate P(Z > 4) for Z ~ N(0, 1)
    """
    print("--- Exercise 3.1: P(Z > 4) ---")
    n = 100000
    
    # Target: Standard Normal
    target_pdf = stats.norm.pdf
    
    # Proposal: Shifted Exponential q(z) = exp(-(z-4)) for z > 4
    # Defined by loc=4 (shift) and scale=1
    proposal = stats.expon(loc=4, scale=1)
    
    # We want to integrate the area z > 4. 
    # Since proposal samples only z > 4, indicator is 1.0.
    target_func = lambda x: 1.0 
    
    res = run_importance_sampling(target_pdf, proposal, target_func, n)
    exact = stats.norm.sf(4)
    
    print(f"Parameters: n={n}, area: [4, inf), proposal: Exp(loc=4)")
    print(f"Exact Value: {exact:.10f}")
    print(f"IS Mean:     {res['mean']:.10f}")
    print(f"Var(Est):    {res['var_estimator']:.2e}")
    print(f"95% CI:      {res['ci']}\n")

# =============================================================================
# EXERCISE 3.2: CAUCHY INTEGRAL WITH SUBSTITUTION AND CONTROL VARIATES
# =============================================================================

def solve_3_2_cauchy_cv():
    """
    Estimates Integral of 1/(pi*(1+x^2)) from 2 to infinity.
    Uses substitution y = 1/x and Control Variates.
    """
    print("--- Exercise 3.2: Cauchy Integral (Substitution + CV) ---")
    n = 100000
    
    # Substitution y = 1/x -> integral of f(y) = 1/(pi*(1+y^2)) over [0, 0.5]
    # Proposal: Uniform over [0, 0.5]
    proposal = stats.uniform(loc=0, scale=0.5)
    y_samples = proposal.rvs(size=n)
    
    # Function to integrate after substitution
    # Note: dy = -1/x^2 dx, but we handle the Jacobian via the uniform pdf constant (0.5)
    def target_f(y):
        return 1 / (np.pi * (1 + y**2))

    # Control Variable: k(y) = 1 - y^2 (Taylor approximation of 1/(1+y^2))
    # We scale k(y) by 1/pi to match target_f
    def control_k(y):
        return (1 - y**2) / np.pi
    
    # Analytical integral of control_k over [0, 0.5]:
    # Int [0, 0.5] (1/pi * (1 - y^2)) dy = 1/pi * [y - y^3/3] from 0 to 0.5
    analyt_integral_k = (1/np.pi) * (0.5 - (0.5**3)/3)
    
    # MC on the difference (f - k)
    # We must account for the density of U(0, 0.5) which is 2.0. 
    # Or simply: Mean(f - k) * length_of_interval
    diff_samples = (target_f(y_samples) - control_k(y_samples)) * 0.5
    
    res_diff = calculate_stats(diff_samples)
    final_mean = res_diff['mean'] + analyt_integral_k
    
    exact = 0.5 - np.arctan(2)/np.pi
    
    print(f"Parameters: n={n}, area: [0, 0.5], CV: (1-y^2)/pi")
    print(f"Exact Value: {exact:.10f}")
    print(f"CV Mean:     {final_mean:.10f}")
    print(f"Var(Est):    {res_diff['var_estimator']:.2e}")
    print(f"95% CI:      ({res_diff['ci'][0] + analyt_integral_k}, {res_diff['ci'][1] + analyt_integral_k})\n")

# =============================================================================
# EXERCISE 3.3: RUIN PROBABILITY (EXPONENTIAL CHANGE OF MEASURE)
# =============================================================================

def simulate_ruin(u, b, mu, n):
    """
    u: upper barrier, b: lower barrier, mu: drift (negative)
    """
    # Finding r such that E[exp(rY)] = 1 for Y ~ N(-mu, 1)
    # Moment generating function of Normal(m, s^2) is exp(mt + 0.5 s^2 t^2)
    # exp(-mu*r + 0.5*r^2) = 1  => -mu*r + 0.5*r^2 = 0 => r = 2*mu
    r = 2 * mu
    
    is_results = []
    for _ in range(n):
        s = 0
        # Under new measure q, drift is +mu
        while -b < s < u:
            s += np.random.normal(mu, 1)
        
        # If we hit upper barrier u, we count the discounted weight
        if s >= u:
            is_results.append(np.exp(-r * s))
        else:
            is_results.append(0.0)
            
    return calculate_stats(np.array(is_results))

def solve_3_3_ruin():
    print("--- Exercise 3.3: Ruin Probability (Rare Event) ---")
    u, b, mu = 10, 10, 0.5
    n = 10000
    
    res = simulate_ruin(u, b, mu, n)
    
    print(f"Parameters: u={u}, b={b}, negative drift mu={-mu}, n={n}")
    print(f"Estimated Ruin Prob: {res['mean']:.6f}")
    print(f"Var(Est):            {res['var_estimator']:.2e}")
    print(f"95% CI:              {res['ci']}\n")

In [ ]:
solve_3_1_tail_probability()
solve_3_2_cauchy_cv()
solve_3_3_ruin()